# NumPyの使い方

NumPy は、同じ種類の数を配列として持ち、形を保ったまままとめて計算するための道具です。Python のリストは値を並べるには便利ですが、足し算、条件抽出、行や列ごとの集計、行列積を何度も行う分析では NumPy の配列が中心になります。

成績表を題材に、1 次元配列、2 次元配列、ブロードキャスト、ブールマスク、集約、行列積、乱数、標準化までを一続きの処理にまとめます。

## 配列は shape と dtype で読む

NumPy の配列を見るときは、値そのものより先に `shape` と `dtype` を確認します。`shape` は配列の形、`dtype` は中身の型です。`(4,)` は 4 個の値が並ぶ 1 次元配列、`(3, 2)` は 3 行 2 列の 2 次元配列を表します。

In [ ]:
import numpy as np


In [ ]:
scores = np.array([72, 88, 65, 91], dtype=np.float64)
score_table = np.array([
    [72, 88, 65],
    [91, 77, 84],
    [68, 79, 95],
])

print("scores:", scores)
print("scores shape:", scores.shape, "dtype:", scores.dtype)
print("score_table:\n", score_table)
print("score_table shape:", score_table.shape, "dtype:", score_table.dtype)


## 配列全体を同じ式で動かす

NumPy では、要素を 1 個ずつ取り出さなくても、配列全体へ同じ計算を適用できます。点数を補正する、割合へ直す、条件を判定する、といった前処理を短く書けます。

In [ ]:
curved = np.minimum(scores + 5, 100)
ratio = scores / 100
passed = scores >= 75

print("curved:", curved)
print("ratio:", ratio)
print("passed:", passed)


同じ処理を Python の `for` で書くと、NumPy が何をまとめて実行しているかが見えます。実務では NumPy の式を使い、意味が分からなくなったときは `for` の形へ戻して確認します。

In [ ]:
curved_with_for = []
for score in scores:
    curved_with_for.append(min(score + 5, 100))

print(curved_with_for)
print(np.array(curved_with_for))
print(np.array_equal(curved, np.array(curved_with_for)))


## 形を変えても要素数は変えない

`reshape` は、要素の数を変えずに並べ方だけを変えます。12 個の値は、`(12,)`、`(3, 4)`、`(2, 2, 3)` のように、積が 12 になる形へ並べ替えられます。

In [ ]:
values = np.arange(1, 13)
grid = values.reshape(3, 4)
blocks = values.reshape(2, 2, 3)

print("values shape:", values.shape)
print(values)
print("grid shape:", grid.shape)
print(grid)
print("blocks shape:", blocks.shape)
print(blocks)


## インデックスとスライスで部分を見る

2 次元配列では、`arr[行, 列]` の順に指定します。行だけを指定すればその行全体、列だけを指定したいときは `:` で行方向をすべて選びます。

In [ ]:
arr = np.arange(1, 13).reshape(3, 4)

print("arr:\n", arr)
print("2行目:", arr[1])
print("3列目:", arr[:, 2])
print("左上2行3列:\n", arr[:2, :3])


## 条件で必要な値を取り出す

比較式を書くと、同じ形の真偽値配列ができます。これをブールマスクとして使うと、条件に合う要素だけを抜き出せます。欠損の除外や外れ値の確認でも同じ考え方を使います。

In [ ]:
mask = score_table >= 80
high_scores = score_table[mask]

print("mask:\n", mask)
print("80点以上:", high_scores)
print("80点以上の数:", high_scores.size)


条件を行単位で使うこともできます。たとえば「どれか 1 科目でも 90 点以上」の学生だけを残すなら、行ごとの条件を作ってから表へ当てます。

In [ ]:
has_excellent_subject = (score_table >= 90).any(axis=1)
selected_students = score_table[has_excellent_subject]

print("row mask:", has_excellent_subject)
print("selected rows:\n", selected_students)


## ブロードキャストで行や列へ同じ補正をかける

ブロードキャストは、形の違う配列を計算できるように NumPy が解釈をそろえる仕組みです。表全体へ同じ列補正を足す、各行から平均との差を引く、といった処理でよく使います。

In [ ]:
subject_bonus = np.array([0, 3, 5])
adjusted = score_table + subject_bonus

print("score_table shape:", score_table.shape)
print("subject_bonus shape:", subject_bonus.shape)
print("adjusted:\n", adjusted)


列方向の平均を引くと、科目ごとに中心を 0 にそろえられます。`keepdims=True` を付けると平均も 2 次元の形を保つため、元の表と引き算しやすくなります。

In [ ]:
subject_mean = score_table.mean(axis=0, keepdims=True)
centered = score_table - subject_mean

print("subject_mean shape:", subject_mean.shape)
print(subject_mean)
print("centered:\n", centered)
print("centered mean:", centered.mean(axis=0))


## axis はどの向きにまとめるかを決める

`axis=0` は行をまたいで列ごとにまとめます。`axis=1` は列をまたいで行ごとにまとめます。結果の `shape` を見ると、どの向きが残ったかを確認できます。

In [ ]:
subject_avg = score_table.mean(axis=0)
student_total = score_table.sum(axis=1)
student_avg = score_table.mean(axis=1)

print("subject_avg:", subject_avg, subject_avg.shape)
print("student_total:", student_total, student_total.shape)
print("student_avg:", student_avg, student_avg.shape)


集約結果は、元の表へ戻して使うこともあります。学生ごとの平均を列として追加すると、表形式の特徴量を作る感覚に近づきます。

In [ ]:
student_avg_column = student_avg.reshape(-1, 1)
score_with_avg = np.concatenate([score_table, student_avg_column], axis=1)

print("student_avg_column shape:", student_avg_column.shape)
print(score_with_avg)


## 行列積は重み付きの合成として読む

`@` は行列積です。機械学習では、入力特徴量に重みを掛けて新しい特徴量やスコアを作る場面で頻繁に出ます。左の行と右の列を対応させて掛け、足し合わせる操作です。

In [ ]:
features = np.array([
    [1.0, 0.2, 0.7],
    [1.0, 0.8, 0.4],
    [1.0, 0.5, 0.9],
])
weights = np.array([
    [0.1, -0.2],
    [0.4, 0.3],
    [0.5, -0.1],
])

logits = features @ weights
print("features shape:", features.shape)
print("weights shape:", weights.shape)
print("logits shape:", logits.shape)
print(logits)


1 行だけ手で計算すると、行列積の中身を確認できます。NumPy の結果と同じになることを見ておくと、`@` を単なる記号ではなく計算として追えます。

In [ ]:
manual_first = [
    features[0, 0] * weights[0, 0] + features[0, 1] * weights[1, 0] + features[0, 2] * weights[2, 0],
    features[0, 0] * weights[0, 1] + features[0, 1] * weights[1, 1] + features[0, 2] * weights[2, 1],
]

print("manual first row:", manual_first)
print("numpy first row:", logits[0])


## 乱数は再現できる形で使う

乱数を使うと、毎回違うサンプルを作れます。学習や検証では、偶然ではなくコードの違いを比較したいことが多いため、`default_rng(seed=...)` で種を固定します。

In [ ]:
rng = np.random.default_rng(seed=42)
samples = rng.normal(loc=50, scale=10, size=(5, 3))

print(np.round(samples, 1))
print("column mean:", np.round(samples.mean(axis=0), 2))
print("column std:", np.round(samples.std(axis=0), 2))


## 標準化で単位の違いをそろえる

平均を引いて標準偏差で割ると、各列の中心が 0、ばらつきが 1 に近づきます。身長と体重のように単位が違う特徴量を、同じモデルへ入れる前処理として使います。

In [ ]:
measurements = np.array([
    [160, 55],
    [170, 68],
    [175, 70],
    [180, 80],
], dtype=np.float64)

mu = measurements.mean(axis=0)
sigma = measurements.std(axis=0)
scaled = (measurements - mu) / sigma

print("mu:", mu)
print("sigma:", np.round(sigma, 2))
print("scaled:\n", np.round(scaled, 2))
print("scaled mean:", np.round(scaled.mean(axis=0), 6))
print("scaled std:", np.round(scaled.std(axis=0), 6))


NumPy の読み方は、`shape` を見る、`axis` を言葉にする、結果が 1 次元か 2 次元かを確かめる、の 3 点に集約できます。配列の形を追えるようになると、Pandas、機械学習、深層学習のコードでも、データがどの向きへ流れているかを見失いにくくなります。